In [ ]:
import os
import json
import random
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

In [ ]:
print(f"TensorFlow version : {tf.__version__}")
print(f"GPU Available      : {tf.config.list_physical_devices('GPU')}")

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# **DATA LOADER**

In [ ]:
TRAIN_DATA_DIR = Path('../../data/train')
TEST_DATA_DIR  = Path('../../data/test')

CLASS_FOLDERS = sorted([d for d in TRAIN_DATA_DIR.iterdir() if d.is_dir()])
CLASS_NAMES   = [d.name for d in CLASS_FOLDERS]
NUM_CLASSES   = len(CLASS_NAMES)

LABEL_MAP = {
    '0_Recyclable': 'Recyclable',
    '1_Electronic':  'Electronic',
    '2_Organic':     'Organic',
}

print(f"Kelas ({NUM_CLASSES}):")
total_train = 0
for folder in CLASS_FOLDERS:
    count = len(list(folder.glob('*')))
    total_train += count
    print(f"  {folder.name:20} => {LABEL_MAP.get(folder.name, folder.name):12} | {count} gambar")

test_count = len(list(TEST_DATA_DIR.glob('*.jpg')))
print(f"\nTotal train : {total_train} gambar")
print(f"Total test  : {test_count} gambar")

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 11))

for row_idx, folder in enumerate(CLASS_FOLDERS):
    image_files = list(folder.glob('*.jpg')) + list(folder.glob('*.jpeg')) + list(folder.glob('*.png'))
    samples = random.sample(image_files[:200], min(5, len(image_files)))
    for col_idx, img_path in enumerate(samples):
        img = Image.open(img_path)
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].set_title(f"{LABEL_MAP.get(folder.name, folder.name)}\n{img.size[0]}x{img.size[1]}", fontsize=8)
        axes[row_idx, col_idx].axis('off')

plt.suptitle('Sample Gambar per Kelas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
class_counts = {LABEL_MAP.get(f.name, f.name): len(list(f.glob('*'))) for f in CLASS_FOLDERS}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(class_counts.keys(), class_counts.values(), color=['steelblue', 'tomato', 'seagreen'], alpha=0.85)

for bar, count in zip(bars, class_counts.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
            str(count), ha='center', va='bottom', fontweight='bold')

ax.set_xlabel('Kelas')
ax.set_ylabel('Jumlah Gambar')
ax.set_title('Distribusi Kelas pada Data Latih', fontweight='bold')
plt.tight_layout()
plt.show()

# **TRAIN-VAL SPLIT**

In [ ]:
PROCESSED_DATA_DIR = Path('../../data/processed')
IMAGE_SIZE         = (224, 224)
TRAIN_RATIO        = 0.85
VAL_RATIO          = 0.15

if not PROCESSED_DATA_DIR.exists():
    raise RuntimeError(
        f"Folder '{PROCESSED_DATA_DIR}' belum ada. Jalankan notebooks/00_data_split.ipynb "
        "dulu untuk generate data/processed/train dan data/processed/val."
    )

for split_name in ['train', 'val']:
    total = sum(
        len(list((PROCESSED_DATA_DIR / split_name / cls).glob('*')))
        for cls in LABEL_MAP.values()
        if (PROCESSED_DATA_DIR / split_name / cls).exists()
    )
    print(f"Total {split_name:5}: {total} gambar")

In [ ]:
TRAIN_DIR = str(PROCESSED_DATA_DIR / 'train')
VAL_DIR   = str(PROCESSED_DATA_DIR / 'val')

ENGLISH_CLASS_NAMES = sorted(LABEL_MAP.values())
print(f"Kelas ({NUM_CLASSES}): {ENGLISH_CLASS_NAMES}")

fig, ax = plt.subplots(figsize=(10, 5))
x     = np.arange(len(ENGLISH_CLASS_NAMES))
width = 0.35

train_counts = [len(list((PROCESSED_DATA_DIR / 'train' / c).glob('*'))) for c in ENGLISH_CLASS_NAMES]
val_counts   = [len(list((PROCESSED_DATA_DIR / 'val'   / c).glob('*'))) for c in ENGLISH_CLASS_NAMES]

ax.bar(x - width/2, train_counts, width, label='Train', color='steelblue', alpha=0.85)
ax.bar(x + width/2, val_counts,   width, label='Val',   color='tomato',    alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(ENGLISH_CLASS_NAMES)
ax.set_ylabel('Jumlah Gambar')
ax.set_title('Distribusi Dataset per Kelas dan Split', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

# **PREPROCESSING**

In [ ]:
BATCH_SIZE = 32

train_augmentation = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
)

eval_normalization = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_augmentation.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=RANDOM_SEED,
)

val_generator = eval_normalization.flow_from_directory(
    VAL_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)

print(f"\nClass indices: {train_generator.class_indices}")

In [ ]:
class_labels_array = train_generator.classes
unique_classes     = np.unique(class_labels_array)

weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=class_labels_array,
)

CLASS_WEIGHTS = dict(zip(unique_classes.tolist(), weights.tolist()))
idx_to_class  = {v: k for k, v in train_generator.class_indices.items()}

print("Class weights (untuk handle imbalance):")
for idx, w in CLASS_WEIGHTS.items():
    print(f"  {idx_to_class[idx]:12} (idx={idx}): weight={w:.4f}")

# **BUILD MODEL**

In [ ]:
def build_mobilenetv2_model(num_classes, input_shape=(224, 224, 3)):
    base_model = MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
    )
    base_model.trainable = False

    model = models.Sequential(name='mobilenetv2_waste_classifier')
    model.add(base_model)
    model.add(layers.Conv2D(256, (3, 3), activation='relu', padding='same', name='custom_conv2d'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name='custom_maxpool'))
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model


model = build_mobilenetv2_model(NUM_CLASSES)
model.summary()

trainable_count = sum(1 for layer in model.layers if layer.trainable)
print(f"\nLayer trainable : {trainable_count} / {len(model.layers)}")

# **TRAINING LOOP**

In [ ]:
MODEL_NAME     = '01_mobilenetv2_tf'
RUN_ID         = datetime.now().strftime('%Y%m%d_%H%M')
RUN_NAME       = f'01_mobilenetv2_tf_{RUN_ID}'
MODEL_BASE_DIR = Path('../../models') / MODEL_NAME
RUN_DIR        = MODEL_BASE_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"RUN_ID  : {RUN_ID}")
print(f"RUN_DIR : {RUN_DIR}")

In [ ]:
LEARNING_RATE              = 1e-3
FINE_TUNE_LR                = 1e-5
EARLY_STOP_PATIENCE_PHASE1  = 2
EARLY_STOP_PATIENCE_PHASE2  = 5
LR_REDUCE_PATIENCE          = 3
LR_REDUCE_FACTOR            = 0.5
MIN_LEARNING_RATE           = 1e-7
CHECKPOINT_PATH     = str(RUN_DIR / 'best_model.keras')
HISTORY_PATH        = str(RUN_DIR / 'training_history.json')
CSV_LOG_PATH        = str(RUN_DIR / 'training_log.csv')
CONFIG_PATH         = str(RUN_DIR / 'config.json')


def make_callbacks(early_stop_patience):
    return [
        callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=early_stop_patience,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ModelCheckpoint(
            filepath=CHECKPOINT_PATH,
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=LR_REDUCE_FACTOR,
            patience=LR_REDUCE_PATIENCE,
            min_lr=MIN_LEARNING_RATE,
            verbose=1,
        ),
        callbacks.CSVLogger(CSV_LOG_PATH, append=True),
    ]


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
print("Phase 1: Feature Extraction (base frozen).")

In [ ]:
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 10
FINE_TUNE_AT  = 100

print("=" * 50)
print("PHASE 1: Feature Extraction")
print("=" * 50)

history_phase1 = model.fit(
    train_generator,
    epochs=PHASE1_EPOCHS,
    validation_data=val_generator,
    callbacks=make_callbacks(EARLY_STOP_PATIENCE_PHASE1),
    class_weight=CLASS_WEIGHTS,
    verbose=1,
)

print("\n" + "=" * 50)
print("PHASE 2: Fine-tuning (unfreeze top MobileNetV2 layers)")
print("=" * 50)

base_model = model.layers[0]
base_model.trainable = True

for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

trainable_in_base = sum(1 for layer in base_model.layers if layer.trainable)
print(f"Trainable layers di base model: {trainable_in_base} / {len(base_model.layers)}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history_phase2 = model.fit(
    train_generator,
    epochs=PHASE2_EPOCHS,
    validation_data=val_generator,
    callbacks=make_callbacks(EARLY_STOP_PATIENCE_PHASE2),
    class_weight=CLASS_WEIGHTS,
    verbose=1,
)

history_combined = {
    'accuracy':     history_phase1.history['accuracy']     + history_phase2.history['accuracy'],
    'val_accuracy': history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy'],
    'loss':         history_phase1.history['loss']         + history_phase2.history['loss'],
    'val_loss':     history_phase1.history['val_loss']     + history_phase2.history['val_loss'],
}

with open(HISTORY_PATH, 'w') as f:
    json.dump(history_combined, f)
print(f"History saved to {HISTORY_PATH}")

num_epochs_trained = len(history_combined['accuracy'])
best_val_accuracy  = max(history_combined['val_accuracy'])
print(f"\nTotal epoch dilatih : {num_epochs_trained}")
print(f"Best val_accuracy   : {best_val_accuracy:.4f}")

# **EVALUATION**

In [ ]:
train_acc_hist  = history_combined['accuracy']
val_acc_hist    = history_combined['val_accuracy']
train_loss_hist = history_combined['loss']
val_loss_hist   = history_combined['val_loss']

phase1_end = PHASE1_EPOCHS

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(16, 6))

ax_acc.plot(train_acc_hist,  label='Train Accuracy', color='darkblue', linewidth=2)
ax_acc.plot(val_acc_hist,    label='Val Accuracy',   color='crimson',  linewidth=2)
ax_acc.axvline(x=phase1_end - 1, color='gray', linestyle='-', linewidth=1, label='Fine-tune start')
ax_acc.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax_acc.set_xlabel('Epoch')
ax_acc.set_ylabel('Accuracy')
ax_acc.legend()
ax_acc.grid(True, alpha=0.3)

ax_loss.plot(train_loss_hist, label='Train Loss', color='darkblue', linewidth=2)
ax_loss.plot(val_loss_hist,   label='Val Loss',   color='crimson',  linewidth=2, linestyle='--')
ax_loss.axvline(x=phase1_end - 1, color='gray', linestyle='-', linewidth=1, label='Fine-tune start')
ax_loss.set_title('Model Loss', fontsize=14, fontweight='bold')
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Loss')
ax_loss.legend()
ax_loss.grid(True, alpha=0.3)

plt.suptitle('Learning Curve (MobileNetV2 Waste Classifier)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
train_loss, train_accuracy = model.evaluate(train_generator, verbose=0)
val_loss,   val_accuracy   = model.evaluate(val_generator,   verbose=0)

print("=" * 50)
print("EVALUASI FINAL")
print("=" * 50)
print(f"Train Accuracy : {train_accuracy * 100:.2f}%")
print(f"Val   Accuracy : {val_accuracy   * 100:.2f}%")
print(f"Train Loss     : {train_loss:.4f}")
print(f"Val   Loss     : {val_loss:.4f}")

In [ ]:
val_generator.reset()

pred_probs  = model.predict(val_generator, verbose=1)
pred_labels = np.argmax(pred_probs, axis=1)
true_labels = val_generator.classes
class_labels = list(val_generator.class_indices.keys())

print("\nClassification Report (Val Set):")
print(classification_report(true_labels, pred_labels, target_names=class_labels))

In [ ]:
conf_matrix            = confusion_matrix(true_labels, pred_labels)
conf_matrix_normalized = conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis]

fig, (ax_count, ax_norm) = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax_count)
ax_count.set_title('Confusion Matrix (Count)', fontweight='bold')
ax_count.set_xlabel('Predicted')
ax_count.set_ylabel('Actual')
ax_count.tick_params(axis='x', rotation=45)

sns.heatmap(conf_matrix_normalized, annot=True, fmt='.2f', cmap='Reds',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax_norm)
ax_norm.set_title('Confusion Matrix (Normalized)', fontweight='bold')
ax_norm.set_xlabel('Predicted')
ax_norm.set_ylabel('Actual')
ax_norm.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# **INFRENECE**

In [ ]:
NUM_SAMPLE_PREDICTIONS = 15
SAMPLES_PER_CLASS      = 5  # 3 kelas x 5 = 15 total

val_samples = []
for cls_name in class_labels:
    cls_dir  = PROCESSED_DATA_DIR / "val" / cls_name
    all_imgs = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.jpeg")) + list(cls_dir.glob("*.png"))
    chosen   = random.sample(all_imgs, min(SAMPLES_PER_CLASS, len(all_imgs)))
    for img_path in chosen:
        val_samples.append((img_path, cls_name))

random.shuffle(val_samples)

batch = np.array([
    preprocess_input(np.expand_dims(np.array(Image.open(p).convert("RGB").resize(IMAGE_SIZE), dtype=np.float32), axis=0))[0]
    for p, _ in val_samples
], dtype=np.float32)

sample_pred_probs = model.predict(batch, verbose=0)

fig, axes = plt.subplots(3, 5, figsize=(18, 11))
axes = axes.flatten()

for i, (img_path, true_label) in enumerate(val_samples):
    pred_label = class_labels[np.argmax(sample_pred_probs[i])]
    confidence = np.max(sample_pred_probs[i]) * 100
    is_correct = true_label == pred_label

    axes[i].imshow(Image.open(img_path).convert("RGB"))
    
    axes[i].set_title(
        f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)",
        color="green" if is_correct else "red",
        fontsize=9,
    )
    axes[i].axis("off")

plt.suptitle("Sample Predictions (Hijau=Benar, Merah=Salah)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# **SAVE CONFIG**

In [ ]:
config = {
    "run_id": RUN_ID,
    "run_name": RUN_NAME,
    "timestamp": datetime.now().isoformat(),
    "model_name": model.name,
    "base_architecture": "MobileNetV2 (imagenet weights, include_top=False)",
    "input_shape": list(model.input_shape[1:]),
    "num_classes": NUM_CLASSES,
    "class_names": ENGLISH_CLASS_NAMES,
    "class_indices": train_generator.class_indices,
    "class_weights": {idx_to_class[int(k)]: v for k, v in CLASS_WEIGHTS.items()},
    "random_seed": RANDOM_SEED,
    "tensorflow_version": tf.__version__,
    "dataset": {
        "train_dir": str(TRAIN_DIR),
        "val_dir": str(VAL_DIR),
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "image_size": list(IMAGE_SIZE),
        "batch_size": BATCH_SIZE,
        "train_samples": train_generator.samples,
        "val_samples": val_generator.samples,
    },
    "augmentation": {
        "rotation_range": 20,
        "width_shift_range": 0.2,
        "height_shift_range": 0.2,
        "shear_range": 0.2,
        "zoom_range": 0.2,
        "horizontal_flip": True,
        "brightness_range": [0.8, 1.2],
        "fill_mode": "nearest",
    },
    "training": {
        "phase1_epochs_configured": PHASE1_EPOCHS,
        "phase2_epochs_configured": PHASE2_EPOCHS,
        "fine_tune_at_layer": FINE_TUNE_AT,
        "phase1_learning_rate": LEARNING_RATE,
        "phase2_fine_tune_learning_rate": FINE_TUNE_LR,
        "optimizer": "Adam",
        "loss": "categorical_crossentropy",
        "epochs_trained_total": num_epochs_trained,
    },
    "callbacks": {
        "early_stopping": {
            "monitor": "val_accuracy",
            "patience_phase1": EARLY_STOP_PATIENCE_PHASE1,
            "patience_phase2": EARLY_STOP_PATIENCE_PHASE2,
            "restore_best_weights": True,
        },
        "reduce_lr_on_plateau": {
            "monitor": "val_loss",
            "factor": LR_REDUCE_FACTOR,
            "patience": LR_REDUCE_PATIENCE,
            "min_lr": MIN_LEARNING_RATE,
        },
        "model_checkpoint": {
            "monitor": "val_accuracy",
            "save_best_only": True,
        },
    },
    "results": {
        "best_val_accuracy": best_val_accuracy,
        "final_train_accuracy": train_accuracy,
        "final_val_accuracy": val_accuracy,
        "final_train_loss": train_loss,
        "final_val_loss": val_loss,
    },
    "artifacts": {
        "best_model_path": CHECKPOINT_PATH,
        "training_history_path": HISTORY_PATH,
        "training_log_csv_path": CSV_LOG_PATH,
    },
}

with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=2)

print(f"Config saved to: {CONFIG_PATH}")
print(json.dumps(config, indent=2))

# **SUBMISSION**

In [ ]:
# train_generator baca dari processed/train/ -> folder: Electronic, Organic, Recyclable
# Competition labels: 0=Recyclable, 1=Electronic, 2=Organic
ENGLISH_TO_COMPETITION_LABEL = {
    "Electronic":  1,
    "Organic":     2,
    "Recyclable":  0,
}

CLASS_INDEX_TO_COMPETITION_LABEL = {
    v: ENGLISH_TO_COMPETITION_LABEL[k]
    for k, v in train_generator.class_indices.items()
}

comp_label_names = {0: "Recyclable", 1: "Electronic", 2: "Organic"}
print("Mapping model index -> competition label:")
for model_idx, comp_label in sorted(CLASS_INDEX_TO_COMPETITION_LABEL.items()):
    print(f"  model idx {model_idx} ({idx_to_class[model_idx]:12}) -> competition label {comp_label} ({comp_label_names[comp_label]})")

In [ ]:
def preprocess_for_inference(image_path, target_size=(224, 224)):
    img = Image.open(image_path).convert('RGB')
    img = img.resize(target_size)
    arr = np.array(img, dtype=np.float32)
    arr = np.expand_dims(arr, axis=0)
    return preprocess_input(arr)


submission_template = pd.read_csv('../../data/template/submission.csv')
test_ids = submission_template['id'].tolist()
print(f"Total test IDs: {len(test_ids)}")
print(f"Contoh IDs: {test_ids[:5]} ... {test_ids[-5:]}")

In [ ]:
test_images_batch = []
valid_ids         = []

print("Loading test images...")
for img_id in test_ids:
    img_path = TEST_DATA_DIR / f'{img_id}.jpg'
    if not img_path.exists():
        img_path = TEST_DATA_DIR / f'{img_id}.png'
    if not img_path.exists():
        print(f"  [WARNING] Image not found: {img_id}")
        continue
    preprocessed = preprocess_for_inference(img_path)
    test_images_batch.append(preprocessed[0])
    valid_ids.append(img_id)

test_array = np.array(test_images_batch, dtype=np.float32)
print(f"Test array shape: {test_array.shape}")

pred_probs_test   = model.predict(test_array, batch_size=BATCH_SIZE, verbose=1)
pred_indices      = np.argmax(pred_probs_test, axis=1)
competition_preds = [CLASS_INDEX_TO_COMPETITION_LABEL[idx] for idx in pred_indices]

print(f"\nPrediksi selesai. Total: {len(competition_preds)}")
dist = Counter(competition_preds)
for label, count in sorted(dist.items()):
    name = comp_label_names[label]
    print(f"  Label {label} ({name:12}): {count} gambar ({count/len(competition_preds)*100:.1f}%)")

### save submission

In [ ]:
submission_df = pd.DataFrame({
    'id':        valid_ids,
    'predicted': competition_preds,
})

submission_path = f'../../data/submission/submission_{MODEL_NAME}_{RUN_ID}.csv'
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Total rows: {len(submission_df)}")
print("\nSample submission:")
print(submission_df.head(10).to_string(index=False))

In [ ]:
NUM_VIZ = 12
sample_indices = random.sample(range(len(valid_ids)), NUM_VIZ)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for plot_idx, sample_idx in enumerate(sample_indices):
    img_id   = valid_ids[sample_idx]
    img_path = TEST_DATA_DIR / f'{img_id}.jpg'
    img_raw  = Image.open(img_path).convert('RGB')

    pred_comp  = competition_preds[sample_idx]
    confidence = np.max(pred_probs_test[sample_idx]) * 100
    pred_name  = comp_label_names[pred_comp]

    axes[plot_idx].imshow(img_raw)
    axes[plot_idx].set_title(
        f'ID: {img_id}\nPred: {pred_name} (label={pred_comp})\nConf: {confidence:.1f}%',
        fontsize=9,
    )
    axes[plot_idx].axis('off')

plt.suptitle('Sample Inference pada Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()